# Preprocess Genomic Data

This notebook is used to read genomic data (in tabular format) from S3 and store features in SageMaker FeatureStore.

## Step 1: Download and read in the genomic dataset

The dataset file `GSE103584_R01_NSCLC_RNAseq.txt` used here was pre-processed using open-source tools and made available on TCIA. It used STAR v.2.3 for alignment and Cufflinks v.2.0.2 for expression calls. Further details can be found in [1]. The original dataset can also be downloaded from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE103584

[1] Zhou, Mu, et al. "Non–small cell lung cancer radiogenomics map identifies relationships between molecular and imaging phenotypes with prognostic implications." Radiology 286.1 (2018): 307-315.

In [2]:
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io, os
from time import gmtime, strftime, sleep
import time
import sagemaker
from sagemaker.session import Session
from sagemaker import get_execution_role
from sagemaker.feature_store.feature_group import FeatureGroup

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\sadegh.etemad\AppData\Local\sagemaker\sagemaker\config.yaml


In [3]:
region = boto3.Session().region_name

boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name='sagemaker', region_name=region)
featurestore_runtime = boto_session.client(service_name='sagemaker-featurestore-runtime', region_name=region)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime
)

role = 'arn:aws:iam::811165582441:role/lung_cancer_diagnostic'
s3_client = boto3.client('s3', region_name=region)

default_s3_bucket_name = feature_store_session.default_bucket()
prefix = 'sagemaker-featurestore-demo'

#### Download the input data from S3

In [4]:
# Get data from S3 
bucket_gen = 'nsclc-clinical-genomic-data-811165582441-eu-west-2-an'

# Genomic data 
data_key_gen = 'GSE103584_R01_NSCLC_RNAseq.txt'

data_location_gen = 's3://{}/{}'.format(bucket_gen, data_key_gen)

gen_data = pd.read_csv(data_location_gen, delimiter = '\t')
gen_data

,Unnamed: 0,R01-023,R01-024,R01-006,R01-153,R01-031,R01-032,R01-033,R01-034,R01-035,...,R01-136,R01-137,R01-138,R01-139,R01-140,R01-141,R01-142,R01-144,R01-145,R01-146
0,1/2-SBSRNA4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A1BG,NaN,2.528510,1.713994,3.143938,1.795080,2.410910,2.538406,NaN,10.386501,...,3.534986,7.560916,3.560950,2.223180,NaN,2.09417,1.944550,NaN,6.130287,NaN
2,A1BG-AS1,NaN,NaN,NaN,0.646213,NaN,NaN,NaN,NaN,NaN,...,2.408296,3.474290,1.382054,2.091330,NaN,1.30469,1.055700,0.939564,1.445220,NaN
3,A1CF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A2LD1,2.034380,0.436761,1.601030,3.366031,0.994382,2.130685,0.842759,1.835353,0.662647,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22121,ZYG11B,3.472997,5.126819,5.544750,3.876170,7.175560,3.422580,6.655650,6.325740,2.384980,...,61.855910,65.592430,114.581090,63.428630,116.916340,94.51021,62.094580,79.505468,71.244270,72.34928
22122,ZYX,11.285434,10.920210,5.967490,26.631100,14.111400,9.634370,10.205300,30.197250,6.857380,...,13.905890,16.761700,7.448490,16.444500,16.349800,8.75406,21.238980,12.383909,24.413600,6.94153
22123,ZZEF1,9.602287,5.065447,5.518227,12.130147,11.410494,12.713140,6.388015,13.301248,6.367440,...,19.809961,6.780926,9.938174,7.529999,10.298610,9.00592,7.571560,2.859420,8.594376,6.71823
22124,ZZZ3,60.781455,50.243009,77.056216,77.235453,81.196830,47.747791,40.879957,110.758786,20.491683,...,63.958158,69.628668,41.601639,83.205648,49.366676,68.39221,71.528003,50.273287,55.254989,95.83870


We will read the txt file and create a subset by removing case IDs and genes that are not relevant.

In [5]:
# Remove case IDs that do not have weight and pack/years in clinical data 
drop_cases = ['R01-003', 'R01-004', 'R01-006', 'R01-007', 'R01-015', 'R01-016', 'R01-018', 'R01-022', 'R01-023', 'R01-098', 'R01-105']

gen_data = gen_data.drop(drop_cases, axis = 1)

# Add column name for genes 
gen_data.rename(columns={'Unnamed: 0':'index_temp'}, inplace=True)

# Transpose the dataframe such that rows = case IDs and cols = genes 
gen_data.set_index('index_temp', inplace=True)
gen_data_t = gen_data.transpose()
gen_data_t.reset_index(inplace=True)
gen_data_t.rename(columns={'index':'Case_ID'}, inplace=True)

We keep the genes suggested in Zhou, Mu, et al. [1]. These are genes corresponding to Metagenes 19, 10, 9, 4, 3, 21 in Table 2 of the paper (https://pubs.rsna.org/doi/pdf/10.1148/radiol.2017161845).

[1] Zhou, Mu, et al. "Non–small cell lung cancer radiogenomics map identifies relationships between molecular and imaging phenotypes with prognostic implications." Radiology 286.1 (2018): 307-315.

In [6]:
# Selecting the columns corresponsind to the subset of genes and case_ID

selected_columns = ['Case_ID','LRIG1', 'HPGD', 'GDF15', 'CDH2', 'POSTN', 'VCAN', 'PDGFRA', 'VCAM1', 'CD44', 'CD48', 'CD4', 'LYL1', 'SPI1', 'CD37', 'VIM', 'LMO2', 'EGR2', 'BGN', 'COL4A1', 'COL5A1', 'COL5A2']

gen_data_t = gen_data_t[selected_columns]

# Replace NaN with 0
data_gen = gen_data_t.fillna(0)


## Step 2: Create SageMaker FeatureStore

Firstly, we cast the object dtype to string which will then map to String feature type in the SageMaker FeatureStore. We add `record_identifier_feature_name` and `event_time_feature_name` columns to the dataset for creating the feature store.

In [7]:
genomic_feature_group_name = 'genomic-feature-group-' + strftime('%d-%H-%M-%S', gmtime())
genomic_feature_group = FeatureGroup(name=genomic_feature_group_name, sagemaker_session=feature_store_session)

current_time_sec = int(round(time.time()))

def cast_object_to_string(data_frame):
    for label in data_frame.columns:
        if data_frame.dtypes[label] == 'object':
            data_frame[label] = data_frame[label].astype("str").astype("string")

# Cast object dtype to string. SageMaker FeatureStore Python SDK will then map the string dtype to String feature type.
cast_object_to_string(data_gen)

# Record identifier and event time feature names
record_identifier_feature_name = "Case_ID"
event_time_feature_name = "EventTime"

# Append EventTime feature
data_gen[event_time_feature_name] = pd.Series([current_time_sec]*len(data_gen), dtype="float64")

# Load feature definitions to the feature group. SageMaker FeatureStore Python SDK will auto-detect the data schema based on input data.
genomic_feature_group.load_feature_definitions(data_frame=data_gen); # output is suppressed


def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")

genomic_feature_group.create(
    s3_uri=f"s3://{default_s3_bucket_name}/{prefix}",
    record_identifier_name=record_identifier_feature_name,
    event_time_feature_name=event_time_feature_name,
    role_arn=role,
    enable_online_store=True
)

wait_for_feature_group_creation_complete(feature_group=genomic_feature_group)

genomic_feature_group.ingest(
    data_frame=data_gen, max_workers=3, wait=True
)

Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
FeatureGroup genomic-feature-group-05-19-10-59 successfully created.


IngestionManagerPandas(feature_group_name='genomic-feature-group-05-19-10-59', sagemaker_fs_runtime_client_config=<botocore.config.Config object at 0x000001FA5CA72A90>, sagemaker_session=<sagemaker.session.Session object at 0x000001FA475B5050>, max_workers=3, max_processes=1, profile_name=None, _async_result=<multiprocess.pool.MapResult object at 0x000001FA5DA80110>, _processing_pool=<pool ProcessPool(ncpus=1)>, _failed_indices=[])

## Next Stage

Next, we'll take a look at preparing the clinical data.

Click here to [continue](./2_preprocess_clinical_data.ipynb).